<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['bieber.txt',
 'janisjoplin.txt',
 'blink-182.txt',
 'adele.txt',
 'disney.txt',
 'eminem.txt',
 'dr-seuss.txt',
 'bob-marley.txt',
 'cake.txt',
 'ludacris.txt',
 'Kanye_West.txt',
 'al-green.txt',
 'nickelback.txt',
 'lady-gaga.txt',
 'paul-simon.txt',
 'johnny-cash.txt',
 'lorde.txt',
 'lil-wayne.txt',
 'prince.txt',
 'notorious-big.txt',
 'britney-spears.txt',
 'notorious_big.txt',
 'lin-manuel-miranda.txt',
 'nicki-minaj.txt',
 'bruno-mars.txt',
 'bjork.txt',
 'dj-khaled.txt',
 'jimi-hendrix.txt',
 'nirvana.txt',
 'nursery_rhymes.txt',
 'missy-elliott.txt',
 'Lil_Wayne.txt',
 'leonard-cohen.txt',
 'kanye.txt',
 'alicia-keys.txt',
 'michael-jackson.txt',
 'dolly-parton.txt',
 'rihanna.txt',
 'dickinson.txt',
 'kanye-west.txt',
 'radiohead.txt',
 'joni-mitchell.txt',
 'bruce-springsteen.txt',
 'amy-winehouse.txt',
 'patti-smith.txt',
 'drake.txt',
 'r-kelly.txt',
 'bob-dylan.txt',
 'beatles.txt']

In [4]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/tmp/ipykernel_24950/3849064916.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [12]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

I0000 00:00:1777202831.829019   24950 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777202831.830310   24950 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777202831.870972   24950 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777202833.124636   24950 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [13]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [14]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [15]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [16]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [17]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [18]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [23]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=50,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 34753.80078125
Loss after epoch 1: 34266.23828125
Loss after epoch 2: 34052.4375
Loss after epoch 3: 34806.6640625
Loss after epoch 4: 35233.71875
Loss after epoch 5: 34561.453125
Loss after epoch 6: 35066.96875
Loss after epoch 7: 34428.53125
Loss after epoch 8: 34121.59375
Loss after epoch 9: 33891.03125
Loss after epoch 10: 34401.03125
Loss after epoch 11: 34282.125
Loss after epoch 12: 33767.96875
Loss after epoch 13: 34118.28125
Loss after epoch 14: 34468.78125
Loss after epoch 15: 33427.0625
Loss after epoch 16: 33577.4375
Loss after epoch 17: 33813.9375
Loss after epoch 18: 33357.5625
Loss after epoch 19: 33256.9375
Loss after epoch 20: 33542.3125
Loss after epoch 21: 33868.9375
Loss after epoch 22: 33425.3125
Loss after epoch 23: 33779.5
Loss after epoch 24: 32927.625
Loss after epoch 25: 33272.6875
Loss after epoch 26: 33490.25
Loss after epoch 27: 32986.0625
Loss after epoch 28: 32789.0625
Loss after epoch 29: 34013.5
Loss after epoch 30: 32262.6875
Loss a

(392246, 719350)

### 4 - Ensayar

In [24]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('since', 0.5240979790687561),
 ('harm', 0.44740524888038635),
 ('pretty', 0.42445066571235657),
 ('send', 0.4121771454811096),
 ('woah', 0.4091184139251709),
 ('sleep', 0.3893686532974243),
 ('right', 0.3877217471599579),
 ('wind', 0.38570624589920044),
 ('here', 0.3838348388671875),
 ('loving', 0.382505863904953)]

In [25]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('back', -0.02587466686964035),
 ('seems', -0.03457274287939072),
 ('must', -0.04149928316473961),
 ('the', -0.054958272725343704),
 ('had', -0.05630103126168251),
 ('years', -0.06042184308171272),
 ('who', -0.06138927862048149),
 ('it', -0.0682104304432869),
 ('sitting', -0.07084158062934875),
 ('break', -0.07433851063251495)]

In [26]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('seven', 0.7470180988311768),
 ('sixty', 0.7040700912475586),
 ('five', 0.5887733101844788),
 ('six', 0.5592243671417236),
 ('feed', 0.5452592372894287),
 ('until', 0.5048712491989136),
 ('feeling', 0.468837171792984),
 ('understand', 0.4688340127468109),
 ('then', 0.46250343322753906),
 ('wisdom', 0.46164199709892273)]

In [27]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[('sunshine', 0.5938575863838196),
 ('buy', 0.5507120490074158),
 ('thing', 0.5410593152046204),
 ('than', 0.5199378728866577),
 ('friend', 0.4863213002681732)]

In [28]:
# Ensayar con una palabra que no está en el vocabulario:
w2v_model.wv.most_similar(negative=["diedaa"])

KeyError: "Key 'diedaa' not present in vocabulary"

In [29]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 2.62175977e-01  1.79775283e-01 -1.74015105e-01  6.90908507e-02
 -2.41422594e-01 -2.78795421e-01 -2.84784913e-01  3.47535312e-01
  5.71768396e-02  9.94658321e-02  6.79556727e-02 -4.17763442e-02
 -2.98152063e-02 -9.14496034e-02 -1.46532342e-01 -9.07289013e-02
  5.69712818e-02  2.63765454e-02 -5.38462168e-03 -6.63222745e-02
 -2.00426936e-01  2.13146940e-01 -3.20814162e-01 -3.42042685e-01
  1.27375320e-01  1.95855215e-01  1.47441640e-01 -3.50304693e-02
 -1.27477437e-01 -2.67867029e-01 -1.55160189e-01  1.40331402e-01
  1.01059958e-01  3.21692824e-01 -1.04469404e-01  6.81582868e-01
  5.16403139e-01  2.01909751e-01  5.77392941e-03  3.53739530e-01
  1.43072680e-01 -9.19722021e-02 -1.65081382e-01  3.01121157e-02
  7.12508932e-02  9.17793065e-03 -2.25092918e-01  1.64988562e-01
  1.59930125e-01 -2.45917156e-01 -2.89840102e-01 -3.62896651e-01
  2.18704239e-01  3.81853133e-01 -3.76418501e-01  3.16198468e-01
  3.51795137e-01  2.81203359e-01  3.29279780e-01  6.47244696e-03
  3.30904305e-01 -1.63305

In [30]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 0.9999999403953552),
 ('babe', 0.4546182453632355),
 ('hope', 0.41716212034225464),
 ('alright', 0.3987375795841217),
 ('new', 0.39582839608192444),
 ('need', 0.3919881284236908),
 ('sad', 0.3842095732688904),
 ('sunshine', 0.3823079764842987),
 ("somethin'", 0.38021162152290344),
 ('real', 0.37453538179397583)]

In [31]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.4546182453632355),
 ('hope', 0.41716212034225464),
 ('alright', 0.3987375795841217),
 ('new', 0.39582833647727966),
 ('need', 0.3919881284236908),
 ('sad', 0.3842095732688904),
 ('sunshine', 0.3823079764842987),
 ("somethin'", 0.38021159172058105),
 ('real', 0.37453532218933105),
 ("ain't", 0.3731948733329773)]

### 5 - Visualizar agrupación de vectores

In [33]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [35]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [ ]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs.
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)